# Three-class sentiment classification with PySpark

Classifies short English texts (app reviews and social-media posts) as **negative**, **neutral** or **positive**
using a distributed Spark ML pipeline: text cleaning, tokenisation, TF-IDF features and multinomial logistic regression.

| Step | What happens |
|------|--------------|
| 1. Setup | Spark session, data paths |
| 2. Load | Partitioned Parquet, train / validation / test splits |
| 3. EDA | Class balance, text length, most frequent words per class |
| 4. Features | Clean text → tokens → stop-word removal → HashingTF → IDF |
| 5. Baseline | Logistic regression in a single `Pipeline` |
| 6. Tuning | 5-fold `CrossValidator` over `regParam` × `elasticNetParam` |
| 7. Evaluation | Confusion matrix, weighted metrics, error analysis, held-out test set |

## 1. Setup

The next cell installs Java, PySpark and downloads the data. It is only needed on a fresh Colab runtime;
on a cluster that already provides Spark and the data, skip it and set `DATA_DIR` below.

In [ ]:
# Colab-only bootstrap: JDK, PySpark and the data folders.
!apt-get install -qq openjdk-17-jdk-headless > /dev/null
!pip install -q pyspark gdown seaborn

import gdown
for folder_id in [
    "1s4yqMI-9KARhuO9uaEUixRhaWjgJarB6",
    "1rmlNAXJWW_zN2RsYQBxlwWqn0wCQt7kz",
    "1wVZ7tH6GzZHcoD9VGCqDZdx-oJDMCiOX",
]:
    gdown.download_folder(f"https://drive.google.com/drive/folders/{folder_id}", quiet=True)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import HashingTF, IDF, RegexTokenizer, StopWordsRemover, StringIndexer
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession, Window, functions as F

sns.set_theme(style="whitegrid")

DATA_DIR = "/content"                      # change for other environments
TRAIN_PATH = f"{DATA_DIR}/nlp train"
VALID_PATH = f"{DATA_DIR}/nlp validation"
TEST_PATH = f"{DATA_DIR}/nlp test"

NUM_FEATURES = 20_000                      # hash space for TF-IDF
SEED = 42
CLASS_NAMES = ["negative", "neutral", "positive"]   # index == `label` column value
CLASS_COLORS = {"negative": "#c0392b", "neutral": "#7f8c8d", "positive": "#27ae60"}

spark = SparkSession.builder.appName("sentiment-classification").getOrCreate()

## 2. Load data

The data is stored as partitioned Parquet; Spark discovers the partition layout automatically.

| Column | Type | Meaning |
|--------|------|---------|
| `id` | int | Unique text id |
| `text` | string | Raw text (model input) |
| `label` | int | Target: 0 = negative, 1 = neutral, 2 = positive |
| `sentiment` | string | Target as a string |
| `subset` | string | `train` / `validation` / `test` |
| `__index_level_0__`, `id_bucket` | int | Technical columns from the export, dropped |

In [ ]:
TECHNICAL_COLS = ["__index_level_0__", "id_bucket"]

df_train = spark.read.parquet(TRAIN_PATH).drop(*TECHNICAL_COLS)
df_valid = spark.read.parquet(VALID_PATH).drop(*TECHNICAL_COLS)
df_test = spark.read.parquet(TEST_PATH).drop(*TECHNICAL_COLS)

df_train.show(5, truncate=80)

## 3. Exploratory data analysis

EDA runs on train + validation. The test split is not touched until the final evaluation.

In [ ]:
df = df_train.unionByName(df_valid).cache()
print(f"Rows (train + validation): {df.count():,}")

class_dist = df.groupBy("sentiment").count().orderBy("sentiment").toPandas()
class_dist["share"] = class_dist["count"] / class_dist["count"].sum()
print(class_dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.bar(class_dist["sentiment"], class_dist["count"], color=[CLASS_COLORS[s] for s in class_dist["sentiment"]])
ax.set_ylabel("Texts")
ax.set_title("Class distribution (train + validation)")
plt.show()

In [ ]:
df = df.withColumn("text_length", F.length("text"))
df.select("text_length").describe().show()

lengths = df.select("text_length").toPandas()["text_length"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=50, color="#1f77b4", edgecolor="white")
ax.set_xlabel("Characters per text")
ax.set_ylabel("Texts")
ax.set_title("Text length distribution")
plt.show()

## 4. Text preprocessing and TF-IDF features

Cleaning: lower-case, keep letters only, collapse whitespace. The reusable `clean_text` helper is applied
identically to every split. The feature pipeline is **fit on the training split only** and then used to
transform the other splits, so no statistics leak from validation or test data.

In [ ]:
def clean_text(frame):
    """Lower-case, strip everything except letters/whitespace, collapse and trim spaces."""
    cleaned = F.lower(F.col("text"))
    cleaned = F.regexp_replace(cleaned, "[^a-z\\s]", " ")
    cleaned = F.regexp_replace(cleaned, "\\s+", " ")
    cleaned = F.trim(cleaned)
    return frame.withColumn("cleaned_text", cleaned)


tokenizer = RegexTokenizer(inputCol="cleaned_text", outputCol="words", pattern="\\s+")
stop_words = StopWordsRemover(
    inputCol="words", outputCol="filtered_words",
    stopWords=StopWordsRemover.loadDefaultStopWords("english"),
)
hashing_tf = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=NUM_FEATURES)
idf = IDF(inputCol="raw_features", outputCol="features", minDocFreq=5)  # ignore very rare terms

feature_pipeline = Pipeline(stages=[tokenizer, stop_words, hashing_tf, idf])

In [ ]:
train_clean = clean_text(df_train)
feature_model = feature_pipeline.fit(train_clean)          # fit on train only

tfidf_train = feature_model.transform(train_clean).cache()
tfidf_valid = feature_model.transform(clean_text(df_valid)).cache()

tfidf_train.select("cleaned_text", "filtered_words", "features").show(3, truncate=60)

In [ ]:
# Most frequent tokens per class (training split)
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=False)
for ax, sentiment in zip(axes, CLASS_NAMES):
    top = (
        tfidf_train.filter(F.col("sentiment") == sentiment)
        .select(F.explode("filtered_words").alias("word"))
        .groupBy("word").count()
        .orderBy(F.desc("count")).limit(20)
        .toPandas()
    )
    ax.barh(top["word"], top["count"], color=CLASS_COLORS[sentiment])
    ax.invert_yaxis()
    ax.set_title(sentiment.capitalize())
    ax.set_xlabel("Occurrences")
fig.suptitle("Top 20 tokens per class (after stop-word removal)")
fig.tight_layout()
plt.show()

The vocabulary overlaps heavily between classes (`app`, `like`, `time`, `day`, `good`), so sentiment has to be
inferred from a handful of discriminative words (`love`, `great`, `happy` vs. `sad`, `bad`, `miss`) and from
the interaction of terms. This is what limits a bag-of-words model.

## 5. Baseline model

A single `Pipeline` holds preprocessing and the classifier, which keeps train and inference
transformations identical. The label indexer is fitted on the training split before being added to the pipeline.
The baseline deliberately uses simpler preprocessing than section 4 (lower-case + `\W` tokenizer, no digit
removal, no `minDocFreq`) so that the effect of the later improvements is visible.

In [ ]:
def evaluate(predictions, label_col="label"):
    """Accuracy and weighted F1 / precision / recall."""
    return {
        name: MulticlassClassificationEvaluator(labelCol=label_col, predictionCol="prediction", metricName=metric).evaluate(predictions)
        for name, metric in [
            ("accuracy", "accuracy"), ("f1", "f1"),
            ("precision", "weightedPrecision"), ("recall", "weightedRecall"),
        ]
    }


baseline_train = df_train.withColumn("text_lower", F.lower("text"))
baseline_valid = df_valid.withColumn("text_lower", F.lower("text"))

label_indexer = StringIndexer(inputCol="sentiment", outputCol="indexedLabel").fit(baseline_train)

baseline_pipeline = Pipeline(stages=[
    RegexTokenizer(inputCol="text_lower", outputCol="tokens_raw", pattern="\\W", gaps=True),
    StopWordsRemover(inputCol="tokens_raw", outputCol="tokens_clean",
                     stopWords=StopWordsRemover.loadDefaultStopWords("english")),
    HashingTF(inputCol="tokens_clean", outputCol="raw_features", numFeatures=NUM_FEATURES),
    IDF(inputCol="raw_features", outputCol="features"),
    label_indexer,
    LogisticRegression(featuresCol="features", labelCol="indexedLabel",
                       family="multinomial", maxIter=20, regParam=0.01),
])

baseline_model = baseline_pipeline.fit(baseline_train)
baseline_pred = baseline_model.transform(baseline_valid)

baseline_metrics = evaluate(baseline_pred, label_col="indexedLabel")
pd.Series(baseline_metrics, name="baseline (validation)").round(4).to_frame()

In [ ]:
# Sample predictions (index order follows the fitted StringIndexer: most frequent class first)
print("Label index order:", label_indexer.labels)
baseline_pred.select("text", "sentiment", "prediction").show(5, truncate=100)

## 6. Hyperparameter tuning

5-fold cross-validation over `regParam` and `elasticNetParam` (L2 vs. L1 penalty), optimised for weighted F1.
Models are trained on the TF-IDF features from section 4.

In [ ]:
lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=10)

param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 1.0])
    .build()
)

cv = CrossValidator(
    estimator=Pipeline(stages=[lr]),
    estimatorParamMaps=param_grid,
    evaluator=MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1"),
    numFolds=5,
    seed=SEED,
)

print(f"Cross-validating {len(param_grid)} configurations x 5 folds on {tfidf_train.count():,} texts ...")
cv_model = cv.fit(tfidf_train)
best_model = cv_model.bestModel

best_lr = best_model.stages[-1]
print(f"Best params: regParam={best_lr.getRegParam()}, elasticNetParam={best_lr.getElasticNetParam()}")

In [ ]:
cv_scores = pd.DataFrame({
    "regParam": [p[lr.regParam] for p in param_grid],
    "elasticNetParam": [p[lr.elasticNetParam] for p in param_grid],
    "cv_f1": cv_model.avgMetrics,
}).sort_values("cv_f1", ascending=False)
cv_scores.round(4)

## 7. Evaluation on the validation set

In [ ]:
predictions = best_model.transform(tfidf_valid).withColumn(
    "pred_label_str",
    F.when(F.col("prediction") == 0.0, "negative").when(F.col("prediction") == 1.0, "neutral").otherwise("positive"),
)

tuned_metrics = evaluate(predictions)
comparison = pd.DataFrame({"baseline": baseline_metrics, "tuned": tuned_metrics}).round(4)
comparison

In [ ]:
ax = comparison.plot.bar(figsize=(7, 3.8), color=["#b0bec5", "#1f77b4"], rot=0)
ax.set_ylim(0, 0.8)
ax.set_title("Baseline vs. tuned model (validation)")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8)
plt.show()

In [ ]:
def plot_confusion(frame, title):
    cm = (
        frame.groupBy("sentiment", "pred_label_str").count().toPandas()
        .pivot(index="sentiment", columns="pred_label_str", values="count")
        .reindex(index=CLASS_NAMES, columns=CLASS_NAMES).fillna(0).astype(int)
    )
    fig, ax = plt.subplots(figsize=(5.6, 4.6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, linewidths=0.5, linecolor="white", ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.show()
    return cm


cm_valid = plot_confusion(predictions, "Confusion matrix, tuned model (validation)")
print("Recall per class:")
print((cm_valid.div(cm_valid.sum(axis=1), axis=0)).round(3))

### Error analysis

Neutral is the pivot class: a large share of both negative (737 of 1,517) and positive (555 of 1,760) texts
are predicted as neutral, while very few cross directly between negative and positive. Short, context-free
posts and sarcasm carry little lexical signal, so a bag-of-words model falls back to the majority-like
neutral class. Examples of misclassified texts:

In [ ]:
(
    predictions.filter(F.col("pred_label_str") != F.col("sentiment"))
    .select("text", "sentiment", "pred_label_str")
    .show(10, truncate=110)
)

## 8. Final evaluation on the held-out test set

The test split has not been used for fitting, feature statistics or model selection. The feature pipeline and
the cross-validated model are applied exactly as they were fitted.

In [ ]:
tfidf_test = feature_model.transform(clean_text(df_test))
test_pred = best_model.transform(tfidf_test).withColumn(
    "pred_label_str",
    F.when(F.col("prediction") == 0.0, "negative").when(F.col("prediction") == 1.0, "neutral").otherwise("positive"),
)

test_metrics = evaluate(test_pred)
pd.Series(test_metrics, name="tuned (test)").round(4).to_frame()

In [ ]:
cm_test = plot_confusion(test_pred, "Confusion matrix, tuned model (test)")

## 9. Limitations and next steps

- **Cross-validation leakage (minor):** TF-IDF statistics (IDF weights) are fitted on the whole training split before
  cross-validation. Moving the feature stages inside the cross-validated `Pipeline` would remove it.
- **Bag-of-words ceiling:** word order, negation and sarcasm are invisible to TF-IDF. Pretrained embeddings or a
  transformer (e.g. Spark NLP) are the natural next comparison.
- **Neutral class:** most errors involve neutral. Class weights or a calibrated decision threshold could be tried.
- **Productionising:** persist the fitted pipeline (`best_model.write().save(...)`), track runs with MLflow, and
  store features / predictions in Delta tables.